# CPET Array Stack — Step-by-Step Walkthrough

<details>
<summary>This notebook makes every stage of the **array-mode** stack (`library/model/modelClass.py`) explicit,</summary>

printing the **inputs and outputs** at each step. It uses the gas CPET model so all buckets
(gas / reactions / membranes / multi-flow) appear.

Pipeline:

```
initialiseModel ──► createModel ──► prepareModel ──► CardioPulmonaryModelArray ──► runSimulationArray
   JSON+meta        name-keyed       flat-vector        the jitted RHS              diffrax.Euler loop
                     objects        "array compiler"                                      │
                                  decodeStates / encodeStates ◄─────────────────────────────┘
```

> Note: a *physiological* run seeds states via `stateSetup.configureStates`
> (baseline/calibration/control), as the production driver `CPET_Run.ipynb` does. Here we start from
> the raw generator init to keep the walkthrough self-contained — so the **mechanics, shapes and
> name↔index mappings are real**, but the short trajectory below is illustrative, not settled.

</details>

In [ ]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'
import jax; jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import numpy as np
np.set_printoptions(suppress=True, precision=4, linewidth=120)

import library.model.modelClass as modelClass
import library.model.modelGen   as modelGen
import library.model.modelEq    as modelEq
print('devices:', jax.devices())

# One config drives the whole walkthrough. control=True so the controllers
# (parameterVariation bucket) are built -> the full 439-state CPET model.
# This walkthrough builds simulationParams inline (no scenario file); the production
# driver CPET_Run.ipynb instead sources these from the scenario's shared.integration.
simulationParams = {
    'gasExchange': {'enabled': True, 'method': 'new'},
    'modelFileName': 'cpet.json',
    'control': True, 'calibration': False,
    # dt is the model's stable explicit-Euler step. Larger values (e.g. 0.01, ~40x)
    # diverge the integration to NaN; 0.00025 (= scenario shared.integration.dt) keeps
    # the run finite. Passed explicitly here so initialiseModel needs no scenario.
    'dt': 0.00025, 'runTime': 1,
    # runsToIgnore = warm-up runs (trajectory discarded); runsToSave = runs kept.
    # Both advance/carry the state. 1+2 actually exercises the multi-run loop in Step 6.
    'runsToIgnore': 1, 'runsToSave': 2,
    'printStatus': True,
}
simulationParams

## Step 1 — `initialiseModel`  (JSON + metadata → `modelStructure`)

<details>
<summary>**In:** `simulationParams` (which model file, gas on/off, control/calibration flags).</summary>

**Out:** `modelStructure` — the parsed model JSON (now **structure only**) enriched with
`data`/`gasRegions` metadata, plus a `configurations.simulationParameters` block **built here at
runtime** from `simulationParams` (`dt`/`dtDense`/`control`/`calibration`/`gasExchange`). The model
JSON no longer carries run flags; if `dt` is `None` it falls back to the scenario's
`shared.integration.dt`.

</details>

In [ ]:
modelStructure = modelClass.initialiseModel(simulationParams)

print('IN : modelFileName =', simulationParams['modelFileName'])
print('OUT: modelStructure top-level keys:')
print('   ', sorted(modelStructure.keys()))
print('   compartments      :', len(modelStructure['compartments']))
print('   connections.*     :', {k: len(v) for k, v in modelStructure['connections'].items()})
print('   reactions         :', len(modelStructure.get('reactions', {})))
print('   simulationParameters:', modelStructure['configurations']['simulationParameters'])

## Step 2 — `createModel`  (`modelStructure` → name-keyed equation objects)

<details>
<summary>Delegates to `modelGen.initModelObjectsNewGasExchange`. This is the **dict-style** model: a bucketed dict of</summary>

equation objects whose reference fields still hold **string names** (e.g. `'V_As'`). This is your inspection handle.

**In:** `simulationParams`, `modelStructure`.
**Out:** `states` (name→value), `modelObjects` (bucketed objects), `structures` (bookkeeping).

</details>

In [ ]:
states, modelObjects, structures = modelClass.createModel(simulationParams, modelStructure)

print('OUT states           :', len(states), 'named states, e.g.')
for k in list(states)[:5]:
    print(f'      {k:14s} = {states[k]}')
print()
print('OUT modelObjects buckets (count of equation objects):')
for b, v in modelObjects.items():
    if isinstance(v, dict) and b not in ('prefixes', 'constants'):
        print(f'      {b:20s} {len(v):4d}', '' if not v else '  classes: ' + ', '.join(sorted({type(o).__name__ for o in v.values()})))
print('      constants            ', len(modelObjects['constants']))

In [ ]:
# A single object BEFORE resolution — note fields hold string NAMES.
import dataclasses
cap = modelObjects['capacitors']['P_As']
print('capacitors["P_As"] is a', type(cap).__module__ + '.' + type(cap).__name__)
for f in dataclasses.fields(cap):
    print(f'   {f.name:22s} = {getattr(cap, f.name)!r}')

## Step 3 — `prepareModel`  (the "array compiler": name-keyed → flat vector)

<details>
<summary>This is the core transformation. It returns three dicts:</summary>

- **`eqDict`**     — the resolved equation objects, bucketed (fields now hold **int indices**).
- **`namesDict`**  — the flat-vector layout: `stateNames`, `name2idx`, per-bucket output **slots**, multi-flow maps.
- **`modelDataDict`** — `initialStates` (flat jnp vector), `constantList` (flat), plus the original name-keyed `states`/`modelObjects` for debugging.

Internally (regions in `prepareModel`): canonical state ordering → enumerate algebraic intermediates →
constants table → **resolve names→indices** → output-slot lists → multi-flow index maps → assemble.

</details>

In [ ]:
eqDict, namesDict, modelDataDict = modelClass.prepareModel(simulationParams, modelStructure)

print('nState (integrated)  :', namesDict['nState'])
print('nExt   (state+algeb) :', namesDict['nExt'], '  ->', namesDict['nExt'] - namesDict['nState'], 'algebraic intermediates')
print('initialStates shape  :', modelDataDict['initialStates'].shape, '(flat jnp vector)')
print('constantList  shape  :', modelDataDict['constantList'].shape)
print()
print('canonical stateNames[:8] :', namesDict['stateNames'][:8])
print('stateNames[-4:]          :', namesDict['stateNames'][-4:], ' <- T/T0 always last')

In [ ]:
# The 42 ALGEBRAIC names (transient flows + partial pressures, NOT integrated)
ext = namesDict['extNames']; n = namesDict['nState']
print('algebraic intermediates (indices %d..%d):' % (n, namesDict['nExt'] - 1))
for nm in ext[n:n+6]:  print('   ', namesDict['name2idx'][nm], nm)
print('    ...')
print()
# name -> index map (sample)
print('name2idx sample:')
for nm in ['P_As', 'V_As', namesDict['stateNames'][0], ext[n]]:
    print(f'   {nm:16s} -> {namesDict["name2idx"][nm]}')

In [ ]:
# The SAME P_As capacitor AFTER resolution — fields are now int indices ...
capR = eqDict['capacitors'][[i for i,k in enumerate(modelObjects['capacitors']) if k=='P_As'][0]]
print('resolved P_As capacitor fields (ints):')
for f in dataclasses.fields(capR):
    print(f'   {f.name:22s} = {getattr(capR, f.name)!r}')

print()
print('... and decoded back to names via dumpResolvedModel (the debug inverse):')
dump = modelClass.dumpResolvedModel(eqDict, namesDict, buckets=['capacitors'], limit=1)
print('  ', dump['capacitors'][0]['class'], dump['capacitors'][0]['fields'])

In [ ]:
# Per-bucket OUTPUT SLOTS: where each bucket scatters its derivatives in the 439-vector
slots = namesDict['slots']
print('output-slot index lists (first few each):')
for key in ['cap', 'ind', 'pv', 'conc', 'rxn', 'conn', 'other']:
    s = slots[key]
    print(f'   {key:6s} n={len(s):4d}  slots[:5]={list(s[:5])}')
print(f'   T slot = {slots["T"]}, T0 slot = {slots["T0"]}')
print()
print('multi-flow maps (string-surgery replaced by precomputed indices):', len(namesDict['multiFlowMaps']), 'resistor(s)')
print('   map[0] =', {k: list(v) for k, v in namesDict['multiFlowMaps'][0].items()})

## Step 4 — `CardioPulmonaryModelArray`  (the jitted RHS)

<details>
<summary>An `eqx.Module` holding the resolved objects + slot maps (passed to `jax.jit` as a **static** arg).</summary>

`__call__(t, y, args)` returns `dy/dt` for the 439 states, computed entirely by integer indexing into a
flat 481-slot working vector — mirroring `models.CardioPulmonaryModel.__call__` term-for-term.

**In:** `t` (scalar), `y` (flat 439-vector), `args` = constants (flat vector).
**Out:** `x` = `dy/dt` (flat 439-vector).

</details>

In [ ]:
cpModel = modelClass.CardioPulmonaryModelArray(eqDict, namesDict, modelStructure)

y0      = modelDataDict['initialStates']     # (439,)
consts  = modelDataDict['constantList']      # (389,)
dydt    = cpModel(0.0, y0, consts)           # (439,)

print('IN : t=0.0,  y0', y0.shape, ' constants', consts.shape)
print('OUT: dy/dt', dydt.shape, ' all finite:', bool(jnp.all(jnp.isfinite(dydt))))
print()
print('a few derivatives, decoded back to names:')
dec = modelClass.decodeStates(dydt, namesDict)
for k in ['P_As', 'V_As', 'T', 'T0']:
    print(f'   d/dt {k:6s} = {dec[k]:+.4f}')

In [ ]:
# return_outputs=True also exposes the 42 ALGEBRAIC intermediates (flows + partial pressures)
dydt2, algeb = cpModel(0.0, y0, consts, return_outputs=True)
print('algebraic intermediates vector:', algeb.shape, '(resistor+membrane+multiflow flows, partial pressures)')
for nm in namesDict['extNames'][namesDict['nState']:namesDict['nState']+4]:
    j = namesDict['name2idx'][nm] - namesDict['nState']
    print(f'   {nm:18s} = {float(algeb[j]):+.4f}')

## Step 5 — `solveModelArray`  (one diffrax.Euler run)

<details>
<summary>Integrates with **the same `diffrax.Euler` + `ConstantStepSize`** the dict oracle uses (this is what gives</summary>

~1e-13 parity), saving on a 100 Hz grid (the downsampling that bounds memory).

**In:** `cpModel`, `runTime`, `states` (flat), `constants`, `dt0`.
**Out:** `ts` (save grid), `ys` (saved trajectory, `[nSave, 439]`), `sts` (final state carried forward).

</details>

In [ ]:
dt0 = modelStructure['configurations']['simulationParameters']['dt']   # the model's real dt (0.00025)
ts, ys, outs, sts = modelClass.solveModelArray(cpModel, 1.0, y0, consts, dt0)

print('IN : runTime=1.0, dt0=%g (model dt),  y0' % dt0, y0.shape)
print('OUT: ts', ts.shape, ' ys', ys.shape, ' outs', outs.shape,
      '(flows on the save grid)  sts', sts.shape)
print()
print('first 3 saved timesteps of a few states (column i of ys = stateNames[i]):')
idx = {n: namesDict['stateNames'].index(n) for n in ['T', 'P_As', 'V_As']}
for step in range(3):
    print('   t=%.2f ' % float(ts[step]), {n: round(float(ys[step, i]), 4) for n, i in idx.items()})
print()
print('outs columns = cpModel.outputNames (resistor + membrane + multi-flow flows):')
oidx = {n: cpModel.outputNames.index(n) for n in cpModel.outputNames[:3]}
for step in range(3):
    print('   t=%.2f ' % float(ts[step]), {n: round(float(outs[step, i]), 4) for n, i in oidx.items()})

## Step 6 — `runSolverArray` / `runSimulationArray`  (the multi-run loop)

<details>
<summary>`runSimulationArray` runs `runsToIgnore` **warm-up** runs (trajectory discarded) then `runsToSave`</summary>

runs (trajectory **kept**), each integrating `runTime` seconds. The final state of every run **seeds
the next** — the clock state `T` keeps accumulating and never resets — and the saved blocks are
concatenated **once** at the end (the memory discipline that keeps peak ~O(saved data)). Results come
back **decoded to a name-keyed dict**, so downstream plotting/processing is unchanged.

The cell below makes the loop visible: with `1` ignored + `2` saved runs of `runTime=1 s`, the kept
trajectory is `2 × saveGrid` long and its clock `T` starts at **1.0** (the warm-up second was run but
dropped) and is continuous across the run boundary.

**In:** flat initial `states`, `cpModel`, `simulationParams`, empty `runsRes`, `structures`, `constants`.
**Out:** `results` (name→trajectory, length `runsToSave × saveGrid`), final flat `states` (carried forward).

</details>

In [ ]:
results, finalStates = modelClass.runSimulationArray(
    modelDataDict['initialStates'], cpModel, simulationParams,
    runsRes={}, structures=modelDataDict['structures'], constants=consts)

T = np.asarray(results['T'])
nIgnore = simulationParams['runsToIgnore']; nSave = simulationParams['runsToSave']
saveGrid = int(round(simulationParams['runTime'] / cpModel.dtDense))   # pts saved per run

print(f'{nIgnore} ignored (warm-up) + {nSave} saved runs, each runTime={simulationParams["runTime"]}s'
      f' on the {cpModel.dtDense}s grid  ->  {saveGrid} pts/run')
print()
print(f'results: name-keyed dict of {len(results)} trajectories, each length {len(T)} = {nSave} x {saveGrid}')
print(f'clock T spans [{T[0]:.2f}, {T[-1]:.2f}]')
print(f'   -> starts at {T[0]:.2f}, not 0.0: the warm-up run advanced the clock '
      f'0->{nIgnore*simulationParams["runTime"]:.0f}s but was DISCARDED')
print(f'   -> saved runs are continuous (no reset) across the boundary @ idx {saveGrid}: '
      f'T[{saveGrid-1}]={T[saveGrid-1]:.2f} -> T[{saveGrid}]={T[saveGrid]:.2f}')
print()
iT = namesDict['stateNames'].index('T')
print(f'finalStates: flat {finalStates.shape}, T={float(finalStates[iT]):.2f}  '
      f'(carried forward to seed the next call -> chainable runs)')
print('sample result keys:', list(results)[:6])

## Step 7 — `decodeStates` / `encodeStates`  (flat ⇄ named round-trip)

<details>
<summary>These are **load-bearing**, not just debug: calibration/control update states **by name** between stages,</summary>

so each stage does `array → decodeStates → mutate by name → encodeStates → array`.

</details>

In [ ]:
named = modelClass.decodeStates(finalStates, namesDict)   # flat -> {name: value}
print('decodeStates: flat 439-vector -> {name: value}. Each name maps to a fixed flat index:')
for nm in ['P_As', 'V_As', 'T']:
    i = namesDict['name2idx'][nm]
    print(f'   {nm:6s}  flat[{i:>3}] = {float(finalStates[i]):10.4f}   ==   named[{nm!r}] = {named[nm]:.4f}')

# encodeStates is the exact inverse (bit-for-bit, it is just a reorder back to canonical order)
roundtrip = modelClass.encodeStates(named, namesDict)
print('encode(decode(x)) == x exactly:', bool(np.array_equal(np.asarray(roundtrip), np.asarray(finalStates))))
print()

# Why it is load-bearing: a control/calibration stage edits states BY NAME between runs, then
# re-encodes to seed the next run. Here we raise the arterial pressure set-point by +50.
print('a control stage edits by name, then re-encodes to seed the next run:')
before = named['P_As']
named['P_As'] = before + 50.0
seed = modelClass.encodeStates(named, namesDict)
i = namesDict['name2idx']['P_As']
print(f'   named["P_As"] {before:.2f} -> {named["P_As"]:.2f}   =>   re-encoded seed[{i}] = {float(seed[i]):.2f}')
print('   (every other state is unchanged, so `seed` is ready to pass straight back into runSimulationArray)')

## Step 8 — Save the run  (`hdf5.schema_sim` → manuBeat-compatible HDF5)

<details>
<summary>The kept `results` + `finalStates` are persisted as a **self-describing run artifact** — the same</summary>

layout the web-app reads:

```
run.h5
├─ attrs: config_id, run_time, dt, return_frequency
├─ model_structure   # the MUTATED runtime structure that ran (incl. configured/calibrated params)
├─ time              # the save grid
├─ raw/<signal>      # every kept trajectory
├─ final_states      # compound: one row, the carried-forward state
├─ processed/<name>/ # post-processed signals (unit/prefix/name attrs)
└─ config/           # ADDITIVE source recipe (model+metadata[+scenario,post_processing]) to re-mount
```

No pickled Python objects (`modelObjects`/`states` blobs are gone), so it is web-portable —
`read_run_result` returns the same `{signals, t, finalStates, processed, units, ...}` payload a
backend serves to a browser. `model_structure` is a *snapshot of what actually ran*; the additive
`config/` group (via `append_config` / `read_config`) holds the *source inputs* so the run can be
re-mounted from the file alone. This is what `CPET_Run.ipynb` does in production (writing to
`notebookData/<name>` and keeping the file); here we write to a temp path, introspect it, and clean up.

**In:** `results`, `finalStates`, `namesDict`, `modelStructure`, + the run's source configs.
**Out:** an `.h5` artifact + the JSON payload read back from it.

</details>

In [ ]:
import json, os, tempfile
from library.hdf5 import schema_sim, engine
import library.utils as utils

# Step 7 gives the named final state; re-decode fresh (the cell above mutated `named`).
namedFinal = modelClass.decodeStates(finalStates, namesDict)
stateNames = list(namedFinal)

runPath = os.path.join(tempfile.gettempdir(), 'walkthrough_run.h5')

# IN: the kept trajectories (results), the time axis, the final state, the model def.
schema_sim.write_run_result(
    runPath,
    runs={k: v for k, v in results.items() if k != 'T'},   # raw/<signal>; T stored as `time`
    t_axis=results['T'],
    state_names=stateNames,
    final_states=[float(namedFinal[n]) for n in stateNames],
    model_structure=json.dumps(modelStructure,
                               default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o)),
    run_params={'runTime': simulationParams['runTime'], 'dt': simulationParams['dt'],
                'returnFrequency': int(round(1 / cpModel.dtDense))},
    config_id=simulationParams['modelFileName'],
)

# a post-processing step appends into /processed/<name> (here: P_As converted Pa->mmHg)
processed = {'P_As_mmHg': {'data': np.asarray(results['P_As']) / 133.322,
                           'metadata': {'unit': 'mmHg', 'prefix': 'P_', 'name': 'Arterial pressure'}}}
schema_sim.append_processed(runPath, 'demo', processed, proc_config={'op': 'unitConversion'})

# /config: the SOURCE inputs this run consumed -> the file alone can re-mount the run.
# This inline walkthrough used a raw model JSON + metadata (no scenario file); CPET_Run.ipynb
# additionally stores the scenario + post-processing config.
schema_sim.append_config(
    runPath,
    configs={'model':    utils.loadJSONfile(utils.configPath('models', simulationParams['modelFileName'])),
             'metadata': utils.loadJSONfile(utils.configPath('metadata.json'))},
    run_config=simulationParams, mode='control')

# OUT: a self-describing file. engine.get_tree -> JSON-serializable hierarchy (web browse).
tree = engine.get_tree(runPath)
print('top-level groups :', [c['name'] for c in tree['children']])

# read it back as the web payload (same shape read_run_result / ResultsEngine.toPayload serve)
payload = schema_sim.read_run_result(runPath)
print('payload keys     :', list(payload))
print('raw signals      :', len(payload['signals']), '| time pts:', len(payload['t']))
print('final_states     :', len(payload['finalStates']), 'states  e.g. P_As =',
      round(payload['finalStates']['P_As'], 3))
print('processed[demo]  :', list(payload['processed']['demo']), '| unit:', payload['units'].get('P_As_mmHg'))
print('attrs            :', payload['metadata'])

# the source recipe, read straight back from the file
cfg = schema_sim.read_config(runPath)
print('config/          :', sorted(cfg), '| mode:', cfg['mode'],
      '| model has', len(cfg['model']), 'top-level keys')

os.remove(runPath)   # walkthrough cleanup; CPET_Run.ipynb keeps it at data/<name>
